In [1]:
print(2)

2


In [1]:
from pathlib import Path
import ipyparallel as ip

NODES = 8  # adjust
client = ip.Client(connection_info=str(Path("~/nfs/security/ipcontroller-client.json").expanduser()))
client.wait_for_engines(NODES - 1)

In [2]:
%%px --local
import sys
print(sys.executable)

/home/gpolovets_google_com/venv/bin/python
[stdout:21] 
/home/gpolovets_google_com/venv/bin/python
[stdout:22] 
/home/gpolovets_google_com/venv/bin/python
[stdout:23] 
/home/gpolovets_google_com/venv/bin/python
[stdout:24] 
/home/gpolovets_google_com/venv/bin/python
[stdout:25] 
/home/gpolovets_google_com/venv/bin/python
[stdout:26] 
/home/gpolovets_google_com/venv/bin/python
[stdout:27] 
/home/gpolovets_google_com/venv/bin/python


In [3]:
%%px --local
import os
print(os.environ['PATH'])

/home/gpolovets_google_com/venv/bin:/home/gpolovets_google_com/.local/bin:/home/gpolovets_google_com/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin
[stdout:21] 
/home/gpolovets_google_com/venv/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin
[stdout:22] 
/home/gpolovets_google_com/venv/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin
[stdout:23] 
/home/gpolovets_google_com/venv/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin
[stdout:24] 
/home/gpolovets_google_com/venv/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin
[stdout:25] 
/home/gpolovets_google_com/venv/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin
[stdout:26] 
/home/gpolovets_google_com/venv/bin:

In [4]:
%%px --local
import dataclasses
from pprint import pformat
from pathlib import Path

import jax
from jax import numpy as jnp
from jax import random
import json
import numpy as np
from etils import epath

jax.config.update("jax_compilation_cache_dir", str(Path("~/.jax_cache").expanduser()))
if not jax.distributed.is_initialized():
    jax.distributed.initialize()
print(jax.devices())

from deepseek_r1_jax import model as dsjax
from deepseek_r1_jax.model import Weights
from deepseek_r1_jax import chkpt_utils as utils

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=2, process_index=1, coords=(2,0,0), core_on_chip=0), TpuDevice(id=3, process_index=1, coords=(3,0,0), core_on_chip=0), TpuDevice(id=6, process_index=1, coords=(2,1,0), core_on_chip=0), TpuDevice(id=7, process_index=1, coords=(3,1,0), core_on_chip=0), TpuDevice(id=8, process_index=2, coords=(0,2,0), core_on_chip=0), TpuDevice(id=9, process_index=2, coords=(1,2,0), core_on_chip=0), TpuDevice(id=12, process_index=2, coords=(0,3,0), core_on_chip=0), TpuDevice(id=13, process_index=2, coords=(1,3,0), core_on_chip=0), TpuDevice(id=10, process_index=3, coords=(2,2,0), core_on_chip=0), TpuDevice(id=11, process_index=3, coords=(3,2,0), core_on_chip=0), TpuDevice(id=14, process_index=3, coords=(2,3,0), core_on_chip=0), TpuD

2025-03-19 04:31:11.258769: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742358671.273085  374951 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742358671.279559  374951 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


[stdout:21] 
[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=2, process_index=1, coords=(2,0,0), core_on_chip=0), TpuDevice(id=3, process_index=1, coords=(3,0,0), core_on_chip=0), TpuDevice(id=6, process_index=1, coords=(2,1,0), core_on_chip=0), TpuDevice(id=7, process_index=1, coords=(3,1,0), core_on_chip=0), TpuDevice(id=8, process_index=2, coords=(0,2,0), core_on_chip=0), TpuDevice(id=9, process_index=2, coords=(1,2,0), core_on_chip=0), TpuDevice(id=12, process_index=2, coords=(0,3,0), core_on_chip=0), TpuDevice(id=13, process_index=2, coords=(1,3,0), core_on_chip=0), TpuDevice(id=10, process_index=3, coords=(2,2,0), core_on_chip=0), TpuDevice(id=11, process_index=3, coords=(3,2,0), core_on_chip=0), TpuDevice(id=14, process_index=3, coords=(2,3,0), core_on_

[stderr:21] 
/home/gpolovets_google_com/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-03-19 04:31:11.146586: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742358671.162336  356709 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742358671.167185  356709 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[stderr:22] 
/home/gpolovets_google_com/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidg

In [5]:
%%px --local
def encode_input(tokenizer, texts, pad_id: int = 0):
    assert isinstance(texts, list)
    inputs = [
        tokenizer.apply_chat_template([{"role": "user", "content": text}]) + tokenizer.encode("<|Assistant|><think>")
        for text in texts
    ]
    max_len = max([len(x) for x in inputs])
    inputs = [(max_len - len(x)) * [pad_id] + x for x in inputs]
    return np.array(inputs)

In [6]:
# BATCH_SIZE=8
BATCH_SIZE=128
# max_seq_len = 512
cfg = dataclasses.replace(dsjax.Config(), max_seq_len=512)
import json
with open("/home/gpolovets_google_com/jax-llm-examples/deepseek_r1_jax/robert_128_short_inputs.json") as f:
    data = json.load(f)
data_on_engine = [data[f"{i+1}"] for i in range(BATCH_SIZE)]

In [8]:
%px
client[:].push({'data_on_engine': data_on_engine})

<AsyncResult(_push): pending>

In [ ]:
%%px --local
# ckpt_path = epath.Path(f"~/bucket/deepseek-r1-jax-chkpt").expanduser()
tokenizer = dsjax.load_tokenizer()
mesh = jax.make_mesh((1, 4, jax.device_count() // 4), ("x", "y", "z"), devices=jax.devices())
cfg = dataclasses.replace(dsjax.Config(), mesh=mesh)
# weights = utils.load_model(epath.Path(ckpt_path).expanduser(), cfg)
weights = Weights.init(random.key(1), cfg)


In [ ]:
# %%px --local
# jax.profiler.stop_trace()

In [ ]:
%%px --local
# input = encode_input(
#     tokenizer,
#     [
#         "Tell me your name",
#         "What is the weather like expressed in long prose in Old English",
#         "Do you like ice cream, be extremely precise",
#     ],
# )
input = encode_input(tokenizer, data_on_engine)
zero_cache = dsjax.KVCache.init(random.key(1), cfg, input.shape[0], cfg.max_seq_len)
jax.debug.print("Starting prefill..")
jax.profiler.start_trace("gs://gpolovets-inference/deepseek/jax/prefill_128/test")
curr_tokens, logits, cache = dsjax.prefill(input, weights, zero_cache, cfg)
jax.block_until_ready(curr_tokens)
jax.profiler.stop_trace()
jax.debug.print("Starting Decode..")
jax.profiler.start_trace("gs://gpolovets-inference/deepseek/jax/decode_128/test")
curr_tokens, tokens_list = curr_tokens[:, cache.length - 1 : cache.length], []
tokens_list = []
LOG_PERIOD=10
jax.profiler.start_trace
for i in range(32):
    print(f"On output token {i}")
    tokens_list.append(curr_tokens)
    curr_tokens, cache = dsjax.decode_step(curr_tokens, weights, cache, cfg)
    jax.block_until_ready(curr_tokens)
    if i == 9:
        jax.profiler.stop_trace()
tokens = np.array(jnp.concatenate(tokens_list, axis=-1))
responses = [tokenizer.decode(row) for row in tokens]
print("Responses:\n" + pformat(responses))

On output token 0
On output token 1
On output token 2
On output token 3
On output token 4
On output token 5
On output token 6
On output token 7
On output token 8
On output token 9
On output token 10
On output token 11
On output token 12
On output token 13
On output token 14
On output token 15
On output token 16
On output token 17
On output token 18
On output token 19
On output token 20
On output token 21
On output token 22
On output token 23
On output token 24
On output token 25
On output token 26
On output token 27
On output token 28
On output token 29
On output token 30
On output token 31
Responses:
[' bac/ad Reformrazilারahabogang lick blaspперⁿابن美元的琥珀 nombres bracket '
 'Combinations Despite contrast�יין Κścia传记 FrancescoFriday� بدون;j曾国 '
 'говigrouprowave',
 ' bac/ad Reformrazilারahabogang lick blaspперⁿابن美元的琥珀 nombres bracket '
 'Combinations Despite contrast�יין Κścia传记 FrancescoFriday� بدون;j曾国 '
 'говigrouprowave',
 ' bac/ad Reformrazilারahabogang lick blaspперⁿابن美元的琥珀 nom

: 